# The Other Half — Take Home
### SkillCred · RAG Interview Bot

In the live session you built the **retrieval** half: load → chunk → embed → compare → find →
ask a grounded question.

This notebook adds the rest:

| Step | What it adds |
|---|---|
| 8 | **Grading** — a second AI call that scores your answer |
| 9 | **Adapting** — the bot gets harder when you do well |
| 10 | **Breaking it** — proving where the quality actually comes from |
| 11 | **The interview loop** — the whole thing, running |
| 12 | **A web app** — the same engine with a proper interface |

**Everything here is pre-written and explained.** You already did the writing.

---

## First — rebuild what you made live

Run this one cell. It's the code you wrote, collected in one place.


In [ ]:
import os, numpy as np
from fastembed import TextEmbedding
from openai import OpenAI

API_KEY    = os.environ.get("OPENAI_API_KEY", "PASTE_YOUR_KEY_HERE")
BASE_URL   = "https://aicredits.in/v1"
CHAT_MODEL = "gpt-4o-mini"
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)


def load_sections(path):
    text = open(path, encoding="utf-8").read()
    sections = []
    for piece in text.split("\n## ")[1:]:
        title, _, body = piece.partition("\n")
        sections.append({"title": title.strip(),
                         "text": body.split("\n---")[0].strip()})
    return sections


def chunk(sections, max_chars=800, min_chars=100):
    chunks = []
    for s in sections:
        parts = s["text"].split("### Deeper layer")
        for i, part in enumerate(parts):
            level = "base" if i == 0 else "deep"
            text = part.strip()
            if i > 0:
                text = text.split("\n", 1)[1].strip()
            for j in range(0, len(text), max_chars):
                piece = text[j:j + max_chars].strip()
                if len(piece) < min_chars:
                    continue
                chunks.append({"title": s["title"], "level": level, "text": piece})
    return chunks


embedder = TextEmbedding(model_name="BAAI/bge-small-en-v1.5",
                         cache_dir="./models", local_files_only=True)


def embed(text):
    return np.array(list(embedder.embed([text]))[0])


def similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


files = [("python_fundamentals.md", None), ("dsa_basics.md", None)]
if os.path.exists("my_notes.md"):
    files.append(("my_notes.md", None))

sections = []
for path, _ in files:
    sections += load_sections(path)

chunks = chunk(sections)
for c, v in zip(chunks, embedder.embed([c["text"] for c in chunks])):
    c["vec"] = np.array(v)

print(f"{len(chunks)} chunks ready ({len(files)} files)")


### And the retrieve function — now with two extra filters

You wrote the core of this. Two arguments are new:

- **`level`** — restrict the search to `"base"` or `"deep"` passages only.
  *This is what makes the bot adaptive in Step 9.*
- **`exclude`** — chunk titles already used, so it never asks the same thing twice.


In [ ]:
def retrieve(question, top_k=1, level=None, exclude=None):
    q = embed(question)
    exclude = exclude or set()

    pool = [c for c in chunks
            if (level is None or c["level"] == level)
            and c["title"] not in exclude]
    if not pool:
        pool = [c for c in chunks if (level is None or c["level"] == level)]

    scored = [(similarity(q, c["vec"]), c) for c in pool]
    scored.sort(key=lambda x: x[0], reverse=True)
    return [c for _, c in scored[:top_k]]


def ask_about(topic, level=None, exclude=None):
    context = retrieve(topic, level=level, exclude=exclude)[0]
    difficulty = "harder, senior-level" if level == "deep" else "base"
    prompt = f"""You are a technical interviewer for an AI/ML role.
Using ONLY the reference material below, ask ONE {difficulty}-level interview question.
Keep it to a single sentence. Do not reveal the answer.

Reference material:
{context['text']}
"""
    r = client.chat.completions.create(model=CHAT_MODEL, temperature=0.5,
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content.strip(), context


print("ready")


---

# Step 8 — Grading an answer

A **second, separate** AI call grades your answer against the *same passage* we retrieved.

Two design points worth understanding:

- **Separate call.** If the same call wrote the question and graded the answer, the AI would be
  marking its own homework.
- **JSON, not prose.** Step 9 has to *act* on the result. An `if` statement can read `"weak"`.
  It cannot read a paragraph of English.

### And it will sometimes fail

You ask for JSON only. The model replies `Sure! Here's the grading: {...}`. `json.loads` needs the
**whole** string to be JSON, so one friendly word crashes your app.

That's not a bug in your code — it's a property of the tool. The `try/except` below cuts out
everything before the first `{` and after the last `}`.


In [ ]:
import json


def score_answer(question, student_answer, context):
    prompt = f"""You are grading a candidate's interview answer.
Reference material (ground truth):
{context['text']}

Question asked: {question}
Candidate's answer: {student_answer}

Grade the answer. Respond ONLY with valid JSON, no other text:
{{"correctness": <0-5 int>, "clarity": <0-5 int>, "feedback": "<one sentence>", "verdict": "<strong or weak>"}}
"""
    r = client.chat.completions.create(model=CHAT_MODEL, temperature=0,
        messages=[{"role": "user", "content": prompt}])

    raw = r.choices[0].message.content.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start, end = raw.find("{"), raw.rfind("}")
        if start != -1 and end != -1:
            try:
                return json.loads(raw[start:end + 1])
            except json.JSONDecodeError:
                pass
        print("Grader didn't return clean JSON:\n", raw)
        return {"correctness": 0, "clarity": 0,
                "feedback": "Could not read the grader.", "verdict": "weak"}


demo_q = "What is the difference between a list and a tuple in Python?"
demo_ctx = retrieve("list tuple", level="base")[0]

print("A SHALLOW ANSWER:")
print(json.dumps(score_answer(demo_q, "A list uses [] and a tuple uses ().", demo_ctx), indent=2))


### Try it with a real answer


In [ ]:
MY_ANSWER = ""      # write a proper answer here

if MY_ANSWER:
    print(json.dumps(score_answer(demo_q, MY_ANSWER, demo_ctx), indent=2))
else:
    print("Write something in MY_ANSWER first.")


---

# Step 9 — Making it adapt

This is what turns a quiz into an *interview*. And it is **one line**.

| Verdict | What happens |
|---|---|
| `"strong"` | search `level="deep"` — same topic, harder material |
| anything else | stay on `level="base"` |

Read that carefully, because it's the point of the whole workshop:

**We do not make the prompt cleverer. We change which material gets searched.**


In [ ]:
def next_question(topic, verdict, exclude=None):
    level = "deep" if verdict == "strong" else "base"      # <- the whole feature
    return ask_about(topic, level=level, exclude=exclude)


sq, sctx = next_question("python lists and tuples", "strong")
wq, wctx = next_question("python lists and tuples", "weak")

print("after STRONG ->", sctx["title"], f"[{sctx['level']}]")
print("after WEAK   ->", wctx["title"], f"[{wctx['level']}]")


---

# Step 10 — Break it on purpose

Claiming "the intelligence is in the retrieval" is easy. Here's the proof.

The two loops below are identical **except the second has no `level` filter.**


In [ ]:
print("WITH the level filter:")
for i in range(4):
    c = retrieve("python lists and tuples", level="deep")[0]
    print(f"   {i+1}. {c['title']:44} [{c['level']}]")

print()
print("WITHOUT it:")
for i in range(4):
    c = retrieve("python lists and tuples")[0]              # no level=
    print(f"   {i+1}. {c['title']:44} [{c['level']}]")


Difficulty became whatever the search happened to return.

**No prompt changed. No model changed.** You changed *which material was searchable*, and the
app's main feature vanished.

> **In a RAG system, most of your quality comes from retrieval — not from clever wording.**

Most people try to fix a RAG app by rewriting the prompt. Usually the real problem is what got
retrieved.


---

# Step 11 — The full interview

Every piece exists. This loop runs them: **ask → listen → grade → decide → repeat.**

Two rules borrowed from real interviews:

- **Never repeat a question.** We track what's been asked in a `set`.
- **Struggling? Move on.** Rephrasing the same question at someone who's stuck helps nobody.

⚠️ A text box appears for your answer. In VS Code it appears at the **top of the window** — if it
looks frozen, look up.


In [ ]:
TOPICS = ["python dictionaries", "python lists and tuples", "big o and time complexity",
          "python generators", "recursion and base cases", "stacks and queues"]


def run_interview(rounds=3):
    print("=== AI Interview Bot ===\n")
    level, topic, asked = "base", TOPICS[0], set()

    for r in range(1, rounds + 1):
        if not [c for c in chunks if c["level"] == level and c["title"] not in asked]:
            asked = set()

        q, ctx = ask_about(topic, level=level, exclude=asked)
        asked.add(ctx["title"])

        print(f"Q{r} [{ctx['level']}] {ctx['title']}: {q}")
        answer = input("Your answer: ")

        score = score_answer(q, answer, ctx)
        print(f"  -> correctness {score['correctness']}/5, clarity {score['clarity']}/5")
        print(f"  -> {score['feedback']}  [{score['verdict']}]\n")

        if score["verdict"] == "strong":
            level = "deep"                           # same topic, deeper
        else:
            level = "base"
            topic = TOPICS[r % len(TOPICS)]          # new topic

    print("=== Interview complete ===")


run_interview(rounds=3)


### Interview yourself on your own notes

If you added `my_notes.md`, put your own topics in the list and run it again.


In [ ]:
TOPICS = ["your topic", "your other topic"]      # <- edit these
# run_interview(rounds=4)


---

# Step 12 — Give it a web interface

Everything so far lives in a notebook. Real users don't open notebooks.

The cell below writes a **Streamlit** app wrapping this exact engine. Streamlit can't run inside a
notebook — it needs its own process. So the cell *writes the file*, and you run it from a terminal:

```
pip install streamlit
streamlit run app.py
```

It opens at `localhost:8501`. It picks up `my_notes.md` automatically, so it interviews you on
your own material too.

**The retrieval and grading code is identical to what you have above.** Only the interface changed.


In [ ]:
%%writefile app.py
"""SkillCred RAG Interview Bot - Streamlit front end."""
import os, json
import numpy as np
import streamlit as st
from openai import OpenAI
from fastembed import TextEmbedding

CHAT_MODEL, EMBED_MODEL = "gpt-4o-mini", "BAAI/bge-small-en-v1.5"
BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://aicredits.in/v1")
st.set_page_config(page_title="AI Interview Bot", page_icon="\U0001F393")


@st.cache_resource                       # load the 67MB model ONCE
def get_embedder():
    if os.path.isdir("./models"):
        return TextEmbedding(model_name=EMBED_MODEL, cache_dir="./models",
                             local_files_only=True)
    return TextEmbedding(model_name=EMBED_MODEL)


def load_sections(path):
    text = open(path, encoding="utf-8").read()
    out = []
    for piece in text.split("\n## ")[1:]:
        title, _, body = piece.partition("\n")
        out.append({"title": title.strip(), "text": body.split("\n---")[0].strip()})
    return out


@st.cache_data                           # parse and embed once
def build_chunks():
    paths = ["python_fundamentals.md", "dsa_basics.md"]
    if os.path.exists("my_notes.md"):
        paths.append("my_notes.md")
    sections = []
    for p in paths:
        sections += load_sections(p)

    chunks = []
    for s in sections:
        parts = s["text"].split("### Deeper layer")
        for i, part in enumerate(parts):
            level = "base" if i == 0 else "deep"
            text = part.strip()
            if i > 0:
                text = text.split("\n", 1)[1].strip()
            for j in range(0, len(text), 800):
                piece = text[j:j + 800].strip()
                if len(piece) < 100:
                    continue
                chunks.append({"title": s["title"], "level": level, "text": piece})

    for c, v in zip(chunks, get_embedder().embed([c["text"] for c in chunks])):
        c["vec"] = np.array(v)
    return chunks


def similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def retrieve(chunks, question, level=None, exclude=None):
    q = np.array(list(get_embedder().embed([question]))[0])
    exclude = exclude or set()
    pool = [c for c in chunks
            if (level is None or c["level"] == level) and c["title"] not in exclude]
    if not pool:
        pool = [c for c in chunks if (level is None or c["level"] == level)]
    return max(pool, key=lambda c: similarity(q, c["vec"]))


def ask(client, ctx, level):
    label = "harder, senior-level" if level == "deep" else "base"
    r = client.chat.completions.create(model=CHAT_MODEL, temperature=0.5,
        messages=[{"role": "user", "content":
            f"You are a technical interviewer. Using ONLY the reference material below, "
            f"ask ONE {label}-level interview question. One sentence. Do not reveal "
            f"the answer.\n\nReference material:\n{ctx['text']}"}])
    return r.choices[0].message.content.strip()


def grade(client, question, answer, ctx):
    r = client.chat.completions.create(model=CHAT_MODEL, temperature=0,
        messages=[{"role": "user", "content":
            f"You are grading a candidate's interview answer.\nReference material:\n"
            f"{ctx['text']}\n\nQuestion: {question}\nAnswer: {answer}\n\n"
            f"Respond ONLY with valid JSON:\n"
            '{"correctness": <0-5>, "clarity": <0-5>, '
            '"feedback": "<one sentence>", "verdict": "<strong or weak>"}'}])
    raw = r.choices[0].message.content.strip().replace("```json", "").replace("```", "")
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        s, e = raw.find("{"), raw.rfind("}")
        if s != -1 and e != -1:
            try:
                return json.loads(raw[s:e + 1])
            except json.JSONDecodeError:
                pass
        return {"correctness": 0, "clarity": 0,
                "feedback": "Could not read the grader.", "verdict": "weak"}


TOPICS = ["python dictionaries", "python lists and tuples", "big o and time complexity",
          "python generators", "recursion and base cases", "stacks and queues"]

st.title("\U0001F393 AI Interview Bot")
st.caption("Questions grounded in your own notes. Built at the SkillCred RAG workshop.")

with st.sidebar:
    api_key = st.text_input("API key", type="password",
                            value=os.environ.get("OPENAI_API_KEY", ""))
    st.caption("Chat model only. Search runs on this machine.")
    if st.button("Start over"):
        for k in ("round", "level", "topic", "asked", "ctx", "question", "history"):
            st.session_state.pop(k, None)
        st.rerun()

if not api_key:
    st.info("Enter your API key in the sidebar to begin.")
    st.stop()

client = OpenAI(api_key=api_key, base_url=BASE_URL)
with st.spinner("Loading your notes..."):
    chunks = build_chunks()
st.caption(f"{len(chunks)} passages loaded")

ss = st.session_state
ss.setdefault("round", 0)
ss.setdefault("level", "base")
ss.setdefault("topic", TOPICS[0])
ss.setdefault("asked", set())
ss.setdefault("history", [])

for h in ss.history:
    with st.chat_message("assistant"):
        st.markdown(f"**Q{h['round']}** *({h['level']} - {h['title']})*")
        st.write(h["question"])
    with st.chat_message("user"):
        st.write(h["answer"])
    with st.chat_message("assistant"):
        st.write(f"Correctness {h['correctness']}/5 - Clarity {h['clarity']}/5")
        st.write(h["feedback"])

if "question" not in ss:
    with st.spinner("Thinking of a question..."):
        ctx = retrieve(chunks, ss.topic, level=ss.level, exclude=ss.asked)
        ss.ctx = ctx
        ss.asked.add(ctx["title"])
        ss.question = ask(client, ctx, ss.level)
        ss.round += 1
    st.rerun()

with st.chat_message("assistant"):
    st.markdown(f"**Q{ss.round}** *({ss.ctx['level']} - {ss.ctx['title']})*")
    st.write(ss.question)

answer = st.chat_input("Your answer...")
if answer:
    with st.spinner("Grading..."):
        s = grade(client, ss.question, answer, ss.ctx)
    ss.history.append({"round": ss.round, "level": ss.ctx["level"],
                       "title": ss.ctx["title"], "question": ss.question,
                       "answer": answer, **s})
    if s["verdict"] == "strong":
        ss.level = "deep"
    else:
        ss.level = "base"
        ss.topic = TOPICS[ss.round % len(TOPICS)]
    del ss.question
    st.rerun()


### Two things worth noticing in that file

- **`@st.cache_resource`** loads the 67 MB model *once*. Without it, the app would reload it every
  time you typed a character.
- **`st.session_state`** remembers your level, topic and history. A web app re-runs the whole
  script on every interaction, so ordinary variables would reset each time. State has to be stored
  deliberately.

---

# What you can now answer in an interview

- Why is grading a **separate** call from asking the question?
- What breaks when you go from 24 chunks to 10,000?
- How would you tell whether *retrieval* or *generation* is your weak point?
- Why force JSON instead of parsing English?
- Why does a `level` filter beat asking the model for "a harder question"?
- How would you stop the grader from just agreeing with the candidate?
- What happens if your chunks are too big? Too small?

Answer those seven and you understand RAG better than most people who list it on a CV.

---

## Things to try

1. Set `max_chars=400` and re-run everything. Does retrieval get better or worse? Why?
2. Add a third `.md` file on a subject you're revising, and interview yourself on it.
3. Change `top_k=1` to `top_k=3` in `ask_about` and pass all three passages to the AI. Better
   questions, or just longer ones?
4. Make the grader stricter. Does your score drop?
